This notebook uses altered neurotorch array and predictor classes that rely on the package tensorstore for indexing the data. This results in only single model iterations of the input and output arrays being stored in memory, as opposed to the entire array, saving immensely on memory usage.

In [ ]:
import tensorstore as ts
import numpy as np
import ac_segmentation.neurotorch.datasets.dataset
import ac_segmentation.neurotorch.core.predictor
import ac_segmentation.neurotorch.nets.RSUNet

TSPredictor = ac_segmentation.neurotorch.core.predictor.TSPredictor
TSArray = ac_segmentation.neurotorch.datasets.dataset.TSArray

In [ ]:
###Load input zarr 

data_in = ts.open({
         'driver':
             'zarr',
         'kvstore':
             'http://bigkahuna.corp.alleninstitute.org/ACdata/Users/kevin/ispim_ome_zarr/H17_x55_S39a_230808_highres/H17_x55_S39a_230808_highres.zarr/highres_Pos89/1/',
     # Use 100MB in-memory cache.
         'context': {
             'cache_pool': {
                 'total_bytes_limit': 100_000_000
             }
         },
         'recheck_cached_data':
         'open',
     })

data_in = await data_in
inarr = data_in[0,0,:,:,:].transpose()
inarr_shape = list(inarr.shape)[::-1]

In [ ]:
###Create output zarr

out_path = 'file:///ACdata/Users/connorl/Example_OutArr.zarr'
data_out = ts.open({
     'driver': 'zarr',
     'kvstore': out_path,
 },
 dtype=ts.float32,
 chunk_layout=ts.ChunkLayout(chunk_shape=[64, 64, 64]),
 create=True,
 shape=inarr_shape).result()

outarr = data_out.transpose()

In [ ]:
###Run segmentation (probability map is written directly to output zarr)

checkpt_file = "/home/russelt/some_ckpt.ckpt"
net = ac_segmentation.neurotorch.nets.RSUNet.RSUNet()
pred = TSPredictor(net, checkpt_file, gpu_device=None)
in_arr = TSArray(inarr)
out_arr = TSArray(outarr)
%time pred.run(in_arr, out_arr, batch_size=80)